# Stage 1 — Run classifiers on the Ethiopia moderation dataset

Produces predictions from three classifiers on the **838 audit-eligible rows** of the cleaned **951-row** corpus (Amharic + Afaan Oromo), saved to `predictions.xlsx` (keyed by `PostID`) for Stage 2.

1. **Perspective API** — generic baseline; reads the **English translation** (Perspective lacks Amharic/Afaan Oromo). API key required.
2. **AfriHate (AfroXLMR)** — fine-tuned here on the AfriHate dataset (no ready-made checkpoint exists); reads native **OriginalText**.
3. **Amharic mBERT** — Amharic-only; reads native **OriginalText**, Amharic rows only.

**Input:** upload `corpus_SCORING_PRIVATE_unmasked.xlsx` (UNMASKED — never publish it).
**Setup:** Runtime → Change runtime type → **T4 GPU**.


## 1. Install dependencies


In [ ]:
!pip -q install transformers datasets torch requests openpyxl pandas scikit-learn --upgrade


In [ ]:
!pip -q install transformers datasets requests openpyxl scikit-learn "pandas==2.2.2"

In [ ]:
!pip install -q -U transformers accelerate datasets

## 2. Upload the unmasked scoring file


In [ ]:
import pandas as pd
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked[1].xlsx'

In [ ]:
import pandas as pd
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked.xlsx'
print('Using file:', CORPUS_PATH)

In [ ]:
import os
print(os.listdir('/content'))

In [ ]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

## 3. Load corpus and select audit-eligible rows


In [ ]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('Reloaded:', len(df), '| persp scored:', df['persp_score'].notna().sum())

In [ ]:
CORPUS_PATH = 'corpus_SCORING_PRIVATE_unmasked[1].xlsx'
TEXT_NATIVE = 'OriginalText'        # AfriHate + Amharic mBERT read this
TEXT_PERSP  = 'EnglishTranslation'  # Perspective reads this
GOLD='Label3Class'; LANG='Language'; AUDIT='IncludeInAudit'; POS_GOLD='Hate'

df = pd.read_excel(CORPUS_PATH, sheet_name='Dataset')
df = df[df[AUDIT]==True].copy()
df = df[df[TEXT_NATIVE].notna()].reset_index(drop=True)
df['gold_bin'] = (df[GOLD].astype(str).str.strip()==POS_GOLD).astype(int)
print(f'Audit-eligible rows: {len(df)} (Hate={df.gold_bin.sum()}, not-Hate={(df.gold_bin==0).sum()})')
print('Languages:', dict(df[LANG].value_counts()))


In [ ]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

## 4. Classifier 1 — Perspective API (English translation)
Set your key when prompted. Unscored rows are kept as `None` (coverage gap).


In [ ]:
import requests, time

PERSPECTIVE_API_KEY = 'PASTE_YOUR_TOKEN_HERE'   # <-- paste your key between the quotes

THRESH = 0.5
URL = 'https://commentanalyzer.googleapis.com/v1alpha1/comments:analyze?key=' + PERSPECTIVE_API_KEY

# --- quick test of ONE call first ---
test = requests.post(URL, json={'comment': {'text': 'I hate you'},
                                'requestedAttributes': {'TOXICITY': {}}}, timeout=30)
print('Test status:', test.status_code)
if test.status_code != 200:
    print('KEY PROBLEM — response:', test.json())
else:
    print('Key works! Running all rows...')
    def persp(t):
        try:
            r = requests.post(URL, json={'comment': {'text': str(t)[:3000]},
                                         'requestedAttributes': {'TOXICITY': {}}}, timeout=30)
            return r.json()['attributeScores']['TOXICITY']['summaryScore']['value'] if r.status_code==200 else None
        except Exception:
            return None
    scores = []
    for i, t in enumerate(df[TEXT_PERSP].tolist()):
        scores.append(persp(t)); time.sleep(1.1)
        if (i+1) % 50 == 0: print(f'  Perspective {i+1}/838')
    df['persp_score'] = scores
    df['persp_pred'] = [None if s is None else int(s>=THRESH) for s in scores]
    print('Done. Perspective scored', df['persp_pred'].notna().sum(), 'of 838')

In [ ]:
df.to_excel('perspective_results.xlsx', index=False)
print('Saved. persp_score rows:', df['persp_score'].notna().sum())
from google.colab import files
files.download('perspective_results.xlsx')

## 5a. Classifier 2 — fine-tune AfroXLMR on AfriHate
There is no ready-made AfriHate classifier, so we fine-tune AfroXLMR on the AfriHate dataset (`afrihate/afrihate`), following Muhammad et al. (2025). We train on the **Amharic** split (AfriHate's Oromo has only a *test* set, so Afaan Oromo predictions rely on cross-lingual transfer — note this in your results). Labels: {hate, abusive, neutral}.

`afro-xlmr-base` is the reliable default on a free T4; switch `BASE_MODEL` to `Davlan/afro-xlmr-large-76L` to match the paper's best (heavier; reduce batch size if you hit out-of-memory).


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
import pandas as pd, numpy as np, torch
from sklearn.metrics import f1_score

BASE_MODEL = 'Davlan/afro-xlmr-base'   # paper-best alt: 'Davlan/afro-xlmr-large-76L'
EPOCHS = 3; MAXLEN = 128; BATCH = 16

# --- load local AfriHate tsv files ---
tr = pd.read_csv('train.tsv', sep='\t')
dv = pd.read_csv('dev.tsv',   sep='\t')
print('train columns:', list(tr.columns))
print('train shape:', tr.shape)
print(tr.head(3))

# --- auto-detect text + label columns ---
TEXTC = next((c for c in ['tweet','text','content'] if c in tr.columns), tr.columns[0])
LABC  = next((c for c in ['label','labels','class'] if c in tr.columns), tr.columns[-1])
print('text col:', TEXTC, '| label col:', LABC)
print('label values:', sorted(tr[LABC].astype(str).unique()))
# --- encode labels (string -> int), keep names for inference ---
LABEL_NAMES = sorted(tr[LABC].astype(str).unique())          # ['Abuse','Hate','Normal']
lab2id = {n:i for i,n in enumerate(LABEL_NAMES)}
print('label mapping:', lab2id)

def to_ds(frame):
    f = frame[[TEXTC, LABC]].dropna().copy()
    f['labels'] = f[LABC].astype(str).map(lab2id)
    return Dataset.from_pandas(f[[TEXTC,'labels']], preserve_index=False)

train_ds_raw, val_ds_raw = to_ds(tr), to_ds(dv)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def tok_fn(b): return tok(b[TEXTC], truncation=True, max_length=MAXLEN)
train_ds = train_ds_raw.map(tok_fn, batched=True)
val_ds   = val_ds_raw.map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(LABEL_NAMES))
model.config.id2label = {i:n for n,i in lab2id.items()}
model.config.label2id = lab2id

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=1)
    return {'macro_f1': f1_score(p.label_ids, pr, average='macro')}

args = TrainingArguments(output_dir='afrihate_model', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32, learning_rate=2e-5,
    fp16=torch.cuda.is_available(), eval_strategy='epoch', save_strategy='no',
    logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()
print('Validation:', trainer.evaluate())

In [ ]:
trainer.save_model('afrihate_model')
tok.save_pretrained('afrihate_model')
print('Model saved to afrihate_model/')

In [ ]:
trainer.save_model('afrihate_model')
tok.save_pretrained('afrihate_model')
# zip it and download to your computer
import shutil
shutil.make_archive('afrihate_model', 'zip', 'afrihate_model')
from google.colab import files
files.download('afrihate_model.zip')
print('Model saved AND downloading as afrihate_model.zip')

In [ ]:
!pip install -q -U transformers accelerate datasets

In [ ]:
import shutil, os
# clear any half-downloaded afrihate cache
cache = os.path.expanduser('~/.cache/huggingface/datasets')
for d in os.listdir(cache) if os.path.exists(cache) else []:
    if 'afrihate' in d.lower():
        shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
        print('cleared:', d)
print('done')

In [ ]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

In [ ]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0], download_mode='force_redownload')

In [ ]:
from huggingface_hub import whoami
from datasets import get_dataset_config_names

# 1) confirm WHO you're logged in as
print("Logged in as:", whoami()["name"])

# 2) test access to the dataset directly
try:
    cfgs = get_dataset_config_names("afrihate/afrihate")
    print("ACCESS OK — configs:", cfgs)
except Exception as e:
    print("ACCESS FAILED:", type(e).__name__, str(e)[:200])

In [ ]:
import shutil, os
cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache):
    for d in os.listdir(cache):
        if 'afrihate' in d.lower():
            shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
            print('cleared:', d)
print('done')

In [ ]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

In [ ]:
from huggingface_hub import login
login(token='PASTE_YOUR_TOKEN_HERE')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import shutil, os
cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache):
    for d in os.listdir(cache):
        if 'afrihate' in d.lower():
            shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
            print('cleared:', d)
print('cache clean')

In [ ]:
print('df rows:', len(df), '| persp scored:', df['persp_score'].notna().sum())

In [ ]:
raw = load_dataset('afrihate/afrihate', TRAIN_LANGS[0])

In [ ]:
from huggingface_hub import logout, login
logout()
login(token='PASTE_YOUR_TOKEN_HERE')

In [ ]:
from huggingface_hub import hf_hub_download
try:
    p = hf_hub_download("afrihate/afrihate", "data/amh/train.tsv", repo_type="dataset")
    print("SUCCESS — token can read gated file:", p)
except Exception as e:
    print("STILL BLOCKED:", str(e)[:250])

In [ ]:
import os
print([f for f in os.listdir('/content') if f.endswith('.tsv')])

In [ ]:
TEXT_NATIVE = 'OriginalText'
LANG = 'Language'
MAXLEN = 128
import torch
print('ready')

## 5b. AfriHate — inference on your corpus (OriginalText)


In [ ]:
AFRIHATE_POSITIVE = {'hate'}   # which AfriHate labels count as Hate (add 'abusive' to include it)
hate_idx = {i for i,n in enumerate(LABEL_NAMES) if n.lower() in AFRIHATE_POSITIVE}
model.eval()
labels, preds = [], []
for i,t in enumerate(df[TEXT_NATIVE].tolist()):
    enc = tok(str(t)[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt').to(model.device)
    with torch.no_grad():
        idx = int(model(**enc).logits.argmax(-1))
    labels.append(LABEL_NAMES[idx]); preds.append(int(idx in hate_idx))
    if (i+1)%100==0: print(f'  AfriHate {i+1}/{len(df)}')
df['afrihate_label']=labels; df['afrihate_pred']=preds
print('AfriHate Hate predictions:', sum(preds))


In [ ]:
df.to_csv('predictions_partial.csv', index=False)
from google.colab import files
files.download('predictions_partial.csv')
print('Saved + downloading. Pred columns:',
      [c for c in df.columns if 'pred' in c or 'score' in c or 'afrihate' in c])

In [ ]:
a = df[df['afrihate_pred'].notna()].copy()
print("=== AfriHate vs your human labels (Hate vs not-Hate) ===")
print("Overall agreement:", f"{(a['afrihate_pred']==a['gold_bin']).mean():.1%}")
print("\nBy language:")
for lang in a[LANG].unique():
    s = a[a[LANG]==lang]
    print(f"  {lang}: {(s['afrihate_pred']==s['gold_bin']).mean():.1%} agreement ({len(s)} rows)")
print("\nRecall — on rows YOU labeled HATE, AfriHate also said Hate:",
      f"{(a[a.gold_bin==1]['afrihate_pred']==1).mean():.1%}")
print("False alarms — on NOT-hate rows, AfriHate said Hate:",
      f"{(a[a.gold_bin==0]['afrihate_pred']==1).mean():.1%}")

print("\n=== CAUGHT (you=Hate, AfriHate=Hate) ===")
for _,r in a[(a.gold_bin==1)&(a.afrihate_pred==1)].head(3).iterrows():
    print(" •", str(r['EnglishTranslation'])[:120])
print("\n=== MISSED (you=Hate, AfriHate=not-Hate) ===")
for _,r in a[(a.gold_bin==1)&(a.afrihate_pred==0)].head(3).iterrows():
    print(" •", str(r['EnglishTranslation'])[:120])

In [ ]:
df.to_excel('predictions_partial.xlsx', index=False)
from google.colab import files
files.download('predictions_partial.xlsx')
print('Saved + downloading. Pred columns:',
      [c for c in df.columns if 'pred' in c or 'score' in c or 'afrihate' in c])

In [ ]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('df:', len(df), '| persp scored:', df['persp_score'].notna().sum())

In [ ]:
import pandas as pd
df = pd.read_excel('perspective_results.xlsx')
print('Reloaded df:', len(df), 'rows | persp scored:', df['persp_score'].notna().sum())

## 6. Classifier 3 — Amharic mBERT (Amharic rows only)
Check the printed `id2label` and adjust `AMMBERT_POSITIVE` so the 'hate' label maps to 1.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

AMM_MODEL = 'amengemeda/amharic-hate-speech-detection-mBERT'
amm_tok = AutoTokenizer.from_pretrained(AMM_MODEL)
amm_model = AutoModelForSequenceClassification.from_pretrained(AMM_MODEL)
amm_model.eval()
print('id2label:', amm_model.config.id2label)   # <-- paste me this

AMM_POSITIVE = {'hate','label_1','1','hate speech'}   # adjust after seeing id2label
hate_ids = {i for i,n in amm_model.config.id2label.items() if str(n).lower() in AMM_POSITIVE}

labels, preds = [], []
for _, row in df.iterrows():
    if str(row[LANG]).strip() != 'Amharic':
        labels.append(None); preds.append(None); continue
    enc = amm_tok(str(row[TEXT_NATIVE])[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt')
    with torch.no_grad():
        idx = int(amm_model(**enc).logits.argmax(-1))
    labels.append(amm_model.config.id2label[idx]); preds.append(int(idx in hate_ids))
df['ammbert_label'] = labels; df['ammbert_pred'] = preds
print('Amharic mBERT scored', df['ammbert_pred'].notna().sum(), 'Amharic rows')

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U transformers

In [ ]:
import pandas as pd, torch
df = pd.read_csv('predictions_partial.csv')
TEXT_NATIVE='OriginalText'; LANG='Language'; MAXLEN=128
print('Reloaded:', len(df), 'rows | cols:', [c for c in df.columns if 'pred' in c or 'score' in c])

In [ ]:
!pip install -q torch torchvision transformers --upgrade --force-reinstall

In [ ]:
TEXT_NATIVE='OriginalText'; LANG='Language'; MAXLEN=128
print('ready')

In [ ]:
!pip install -q -U transformers

In [ ]:
import pandas as pd, torch
df = pd.read_csv('predictions_partial.csv')
print('Reloaded:', len(df), 'rows | cols:', [c for c in df.columns if 'pred' in c or 'score' in c])

## 7. Save predictions for Stage 2


In [ ]:
keep=['PostID',LANG,GOLD,'gold_bin','persp_score','persp_pred',
      'afrihate_label','afrihate_pred','ammbert_label','ammbert_pred']
out=df[[c for c in keep if c in df.columns]].copy()
out.to_excel('predictions.xlsx', index=False)
print('Saved predictions.xlsx with', len(out), 'rows.')
from google.colab import files; files.download('predictions.xlsx')


In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

df = pd.read_csv('predictions_partial.csv')
LANG = 'Language'
CLASSIFIERS = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred')]

def metrics_row(name, scope, frame, pred_col):
    sub = frame[frame[pred_col].notna()]
    if len(sub) == 0: return None
    yt = sub['gold_bin'].astype(int); yp = sub[pred_col].astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
    return {'classifier':name, 'scope':scope,
            'coverage':f"{len(sub)}/{len(frame)} ({len(sub)/len(frame):.0%})",
            'accuracy':round(accuracy_score(yt,yp),3),
            'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),
            'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}

rows = []
for name, col in CLASSIFIERS:
    rows.append(metrics_row(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r = metrics_row(name, lang, df[df[LANG]==lang], col)
        if r: rows.append(r)
results = pd.DataFrame([r for r in rows if r])

print("="*72)
print("STAGE 2 — CLASSIFIER EVALUATION  (positive class = Hate)")
print("="*72)
print(results.to_string(index=False))

print("\n--- Perspective coverage by language ---")
for lang in ['Amharic','Afan Oromo']:
    s = df[df[LANG]==lang]; sc = s['persp_pred'].notna().sum()
    print(f"  {lang}: {sc}/{len(s)} scored ({sc/len(s):.0%})")

results.to_csv('stage2_results.csv', index=False)
from google.colab import files
files.download('stage2_results.csv')
print("\nSaved + downloading stage2_results.csv")